# 1. EDA — Biohub Cell Tracking During Development

Purpose: understand the OME-Zarr + GEFF data format, survey shape/scale/
ground-truth density across multiple training videos (not just one), and
inspect a single video in depth (frame visualization, division timing)
before designing/tuning a model.

Not yet run: no data is present in this dev environment (dataset is kept
off local disk — see `docs/0_coding_standards.md`). Run this on Kaggle via
`scripts/push_kaggle_kernel.sh eda` (the competition mount and the public
`cellmot-baseline-artifacts` dataset, which bundles a working `repo/`, both
auto-detect — see the Setup cell) or locally after downloading one sample
and pointing `$CELLMOT_DATA_DIR` at it.

## 1. Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

IS_KAGGLE = Path("/kaggle").exists()


def _find_mount(candidates: list[Path], marker: str) -> Path | None:
    """Return the first candidate containing ``marker``, else scan /kaggle/input."""
    for c in candidates:
        if (c / marker).exists():
            return c
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for p in kaggle_input.glob(f"**/{marker}"):
            return p.parent
    return None


if IS_KAGGLE:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "zarr>=3.0.10", "scipy", "tqdm", "polars",
            "tracksdata @ git+https://github.com/royerlab/tracksdata@main",
        ],
        check=True,
    )
    ARTIFACTS_MOUNT = _find_mount(
        [
            Path("/kaggle/input/cellmot-baseline-artifacts"),
            Path("/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts"),
        ],
        "weights",
    )
    if ARTIFACTS_MOUNT is None:
        raise FileNotFoundError(
            "cellmot-baseline-artifacts dataset not found under /kaggle/input -- add "
            "it as a data source (see kernel-metadata.json)."
        )
    REPO_ROOT = ARTIFACTS_MOUNT / "repo"  # read-only is fine -- EDA never writes predictions/weights
else:
    REPO_ROOT = Path.cwd().parent

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "scripts"))

from dataspec import DATASET_PATH  # noqa: E402 -- needs REPO_ROOT on sys.path first

from tracking_cellmot.io import list_datasets, open_dataset  # noqa: E402

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR = Path(f"/kaggle/input/competitions/{COMPETITION}")
TEST_DIR = COMP_DIR / "test" if IS_KAGGLE else REPO_ROOT / "data" / "test"

SEED = 0
np.random.seed(SEED)

DATASET_NAME = None      # None -> pick the first available training dataset
FRAME_INDEX = 0           # timepoint to visualize for the single-dataset deep dive
N_VIDEOS_FOR_STATS = 10   # how many train videos to summarize in the stats table; None = all

print(f"IS_KAGGLE={IS_KAGGLE}  REPO_ROOT={REPO_ROOT}  DATASET_PATH={DATASET_PATH}  TEST_DIR={TEST_DIR}")

## 2. List available datasets (train + test)

In [ ]:
train_datasets = list_datasets(DATASET_PATH, require_geff=True)
print(f"DATASET_PATH = {DATASET_PATH}")
print(f"{len(train_datasets)} train datasets with ground truth found")
for p in train_datasets[:10]:
    print(" -", p.stem)

test_datasets = list_datasets(TEST_DIR, require_geff=False) if TEST_DIR.exists() else []
print(f"\nTEST_DIR = {TEST_DIR}")
print(f"{len(test_datasets)} test datasets found (no ground truth, per the competition)")
for p in test_datasets[:10]:
    print(" -", p.stem)

## 3. Multi-video stats table (train)

A single video isn't representative — summarize shape, scale, and
ground-truth density across the first `N_VIDEOS_FOR_STATS` train videos so
modeling decisions (e.g. `--unet-batch-size`, `--det-threshold` in
`02_baseline_modeling.ipynb`) are grounded across the dataset, not one clip.

In [ ]:
import polars as pl

survey = train_datasets[:N_VIDEOS_FOR_STATS] if N_VIDEOS_FOR_STATS else train_datasets

rows = []
for p in survey:
    ds_i = open_dataset(p, normalize=False, require_tracks=True)
    out_degree_i = ds_i.tracks.edge_attrs().group_by("source_id").len()
    n_t = ds_i.image.shape[0]
    n_nodes = ds_i.tracks.num_nodes()
    rows.append({
        "dataset": p.stem,
        "T": n_t,
        "Z": ds_i.image.shape[1],
        "Y": ds_i.image.shape[2],
        "X": ds_i.image.shape[3],
        "dtype": str(ds_i.image.dtype),
        "scale_z_y_x_um": tuple(round(s, 4) for s in ds_i.scale),
        "nodes": n_nodes,
        "edges": ds_i.tracks.num_edges(),
        "divisions": int((out_degree_i["len"] == 2).sum()),
        "nodes_per_timepoint": round(n_nodes / n_t, 2),
    })

stats = pl.DataFrame(rows)
print(f"Surveyed {len(survey)}/{len(train_datasets)} train videos")
stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(stats["dataset"], stats["nodes_per_timepoint"], color=plt.cm.viridis(0.5))
axes[0].set_title("Annotated nodes per timepoint")
axes[0].tick_params(axis="x", rotation=90)
axes[1].bar(stats["dataset"], stats["divisions"], color=plt.cm.viridis(0.8))
axes[1].set_title("Division events")
axes[1].tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

*Insight: fill in after running — do shape (T/Z/Y/X) and annotation density
vary a lot across videos (affects whether one `--unet-batch-size` /
`--det-threshold` fits all), and are divisions rare enough that
`ILP_DIVISION_WEIGHT` in `02_baseline_modeling.ipynb` needs tuning per the
observed rate?*

## 4. Deep dive: one dataset in detail

In [ ]:
name = DATASET_NAME or train_datasets[0].stem
ds = open_dataset(DATASET_PATH / name, normalize=False, require_tracks=True)

print(f"dataset:      {name}")
print(f"image shape:  {ds.image.shape}  (T, Z, Y, X)")
print(f"image dtype:  {ds.image.dtype}")
print(f"voxel scale:  {ds.scale}  microns (Z, Y, X)")
print(f"track nodes:  {ds.tracks.num_nodes()}")
print(f"track edges:  {ds.tracks.num_edges()}")

*Insight: fill in after running — how many timepoints, how sparse are the
annotations relative to the number of cells visible in the image, does
sparsity vary across the video.*

### Visualize one frame with annotated cell centers

In [ ]:
frame = np.asarray(ds.image[FRAME_INDEX])
mip = frame.max(axis=0)  # max-intensity projection over Z

node_attrs = ds.tracks.node_attrs()
frame_nodes = node_attrs.filter(node_attrs["t"] == FRAME_INDEX)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(mip, cmap="viridis")
ax.scatter(frame_nodes["x"], frame_nodes["y"], s=12, facecolors="none", edgecolors="white", linewidths=0.8)
ax.set_title(f"{name} — frame {FRAME_INDEX} — {frame_nodes.height} annotated cells")
ax.axis("off")
plt.show()

*Insight: fill in after running.*

### Division timing (this dataset)

In [ ]:
edge_attrs = ds.tracks.edge_attrs()
out_degree = edge_attrs.group_by("source_id").len()
n_divisions = (out_degree["len"] == 2).sum()
print(f"division events (source with 2 outgoing edges): {n_divisions}")
print(f"total timepoints: {ds.image.shape[0]}")

## 5. Findings / limitations / next experiment

- **Findings**: _fill in after running._
- **Limitations**: `N_VIDEOS_FOR_STATS` caps the multi-video survey (default
  10, of however many are found) for a fast first pass — rerun with `None`
  for the full dataset once the notebook's runtime on Kaggle is known.
- **Next**: run `02_baseline_modeling.ipynb` (`RUN_MODE = "submission"`,
  `USE_PRETRAINED = True`) to get a first submission out, then revisit EDA
  on the videos where the model underperforms.